<!-- DATA PROVIDER INSTRUCTIONS

1. Provide the name of your dataset, replacing the bracketed placeholder text.
2. Update the Registry of Open Data landing page URL, by replacing the bracketed placeholder text. The [REGISTRY_YAML_NAME] will correspond to the name of the YAML document in your pull request to the Registry of Open Data on Github, minus the .yaml file extension.
3. Remove these comment blocks when you have completed each section.

DATA PROVIDER INSTRUCTIONS -->

# Get to Know a Dataset: [ESO137-006]

This notebook serves as a guided tour of the [[ESO137-006]](https://registry.opendata.aws/[REGISTRY_YAML_NAME]) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

<!-- DATA PROVIDER INSTRUCTIONS

The goal of this section is to orient users to the structure of your dataset. 

1. How are key prefixes and objects organized in your S3 bucket?
2. What kinds of filetypes are represented in your dataset?
3. Explain with text what users are expected to encounter, and then demonstrate with code the organizational framework you applied when creating your dataset.
4. The responses to each question section are meant to be expanded or replaced as dictated by your dataset

DATA PROVIDER INSTRUCTIONS -->

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

EXAMPLE - REPLACE

At the top level of our S3 bucket, we have a single key prefix "ESO137" that in turn contains:

 1. the "ms1_primary.zarr" single key prefix containing Measurement Set v2 data 
 2. the "ms1_target.zarr" single key prefix containing Measurement Set v2 data
 3. the "ms2_target.zarr" single key prefix containing Measurement Set v2 data.
 
 Full documentation for this dataset can be found at: https://www.aanda.org/articles/aa/abs/2020/04/aa37800-20/aa37800-20.html.

EXAMPLE - REPLACE



In [1]:
# CODING GUIDELINES FOR DATA PROVIDER
#
# General notebook coding guidelines:
# 1. Assume that your reader understands the basics of Jupyter Notebooks, Python, and their Python environment.
#    The focus of this tutorial is on your dataset.
# 2. For library requirements, list the required libraries in a comment block in "requirements.txt" format
#    (https://pip.pypa.io/en/stable/reference/requirements-file-format/)
# 3. Demonstrate importing libraries with the assumption that the user has correctly installed the required
#    libraries.
# 4. List and load all library dependencies once, at this point of the notebook, unless a complicated dependency
#    set makes it unweildy.
# 5. Remember, the goal of this tutorial is a 101-level introduction to your dataset using common tools and libraries.
#    Examples using specialized environments and deep-diving methods are better suited to follow-up tutorials.
#
# CODING GUIDELINES FOR DATA PROVIDER

First we will import the Python libraries required throughout this notebook.


In [2]:
### EXAMPLE - REPLACE


# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# dask-ms[s3,zarr] > 0.2.30
# datashader >= 0.19.0
# pillow >= 12.2.0
# xarray >= 2026.4.0

# Import the libraries required for this notebook
# Built-ins
from functools import partial
import json
from pprint import pprint
# Installed libraries
from dask.diagnostics import ProgressBar
import dask.array as da
import dask.dataframe as dd
from daskms import xds_from_storage_ms
import numpy as np
import xarray

import datashader
import datashader.transfer_functions as transfer_functions

### EXAMPLE - REPLACE


Next, we will define the location of our zarr dataset containing the ESO137-006 data.


In [3]:
# Location of the S3 bucket for this dataset
bucket = "ratt-public-data"


<!-- DATA PROVIDER INSTRUCTIONS
This section is meant to orient users of your dataset to the formats present in your dataset, particularly if your dataset includes formats that may be unfamiliar to a general data scientist audience. This section should include:

1. Explanation of data format(s) (very common formats can be very briefly described, while less common
   or domain specific formats should include more explanation as well as links to official documentation)
2. Explanation of why the data format was chosen for your dataset
3. Recommendations around software and tooling to work with this data format
4. Explanation of any dataset-specific aspects to your usage of the format
5. Description of AWS services that may be useful to users working with your data
DATA PROVIDER INSTRUCTIONS -->

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

EXAMPLE - REPLACE
Our dataset is structured as Hierarchical [Zarr](https://zarr.dev/) Groups, each exposed as an [Xarray](https://xarray.dev/) dataset.  Each dataset contains a *scan* of the MeerKAT telescope where data representing the correlated electronic voltages (visibilities) are recorded at multiple timesteps and frequencies, between antenna pairs (baselines).

Our dataset used this format because [Zarr](https://zarr.dev/):
 - is a chunked storage format suitable for storing large quantities of data on both Cloud Object Stores and Posix File Systems.
 - integrates with both [NumPy](https://numpy.org/) and [Xarray](https:/xarray.dev) to provides a convenient interface for working with the underlying data.
 - handles various NumPy data types well (coordinates, timestamps, visibility data).
 - is easily processed by most programming languages.

These Zarr groups primarily contain electronic voltage data (DATA and CORRECTED_DATA), their associated weights (WEIGHT_SPECTRUM) and a mask (FLAG) indicating bad or missing data. Secondary data includes antenna coordinate positions, names and offsets as well as directional information on the field and source under observation. The data in these groups is more fully document in the [Measurement Set v2.0](https://casa.nrao.edu/Memos/229.html) specification.

Zarr is well supported by Python through its built in 'zarr-python' library. Packages such as Pandas and Polars can be used to work with JSON as well.

EXAMPLE - REPLACE

<!-- DATA PROVIDER INSTRUCTIONS
The goal of this section is to demonstrate loading a portion of data from your dataset, and reveal something about its structure.
1. Load an object from S3
2. Show the structure of data in the object
DATA PROVIDER INSTRUCTIONS -->

### Q: Can you show us an example of downloading and loading data from your dataset?

As an example, let us load up and look into some visibility data found in s3://ratt-public-data/ESO137/ms1_target.zarr.


In [ ]:

# Location of the ESO137 Zarr visibility dataset
url = f"s3://{bucket}/ESO137/ms1_target.zarr"
# Load xarray dataset metadata from the remote zarr dataset
# via dask-ms [dask-ms](https://dask-ms.readthedocs.io/en/latest/).
datasets = xds_from_storage_ms(url)

for ds in datasets:
  print(ds)

Our observation has been subdivided into 7 different datasets, each containing a scan of about 460GB of data. The format is tabular, but with secondary dimensions. Each row represents a measurement taken between two antenna (ANTENNA1 and ANTENNA2) at a particular time (TIME) across multiple channels (chan) and correlations (corr). The UVW variable contains the U, V and W coordinates of the baseline. Uncompressed, these datasets represent almost 3TB of data.


In [ ]:
print(f"Total Dataset Size {sum(ds.nbytes for ds in datasets) / (1024.**4):0.2f}TB")

Lets work out the number of unique antenna in a dataset using dask

In [ ]:
a1 = datasets[0].ANTENNA1.data
a2 = datasets[0].ANTENNA2.data

unique_antenna = da.unique(da.concatenate([a1, a2])).compute()
uants = unique_antenna.shape[0]
print(f"{uants} antenna")

And the number of unique baselines (antenna pairs)

In [ ]:
baselines = da.stack([a1, a2], axis=1).rechunk((-1, 2))
# da.unique doesn't take an axis argument so we construct a parallel reduction by hand
unique_baselines = da.reduction(
  baselines,
  concatenate=True,
  chunk=lambda data, axis, keepdims: np.unique(data, axis=0),
  aggregate=lambda data, axis, keepdims: np.unique(data, axis=0),
  output_size=np.nan,
  axis=0,
  meta=np.empty((0, 0), dtype=baselines.dtype),
  keepdims=True,
  dtype=baselines.dtype).compute()

print(
  f"{unique_baselines.shape[0]} unique baselines.\n"
  f"{uants * (uants + 1) // 2} baselines expected if matching antenna (auto-correlations) are included\n"
  f"{uants * (uants - 1) //2} baselines expected if matching antenna (auto-correlations) are excluded\n"
)


<!-- DATA PROVIDER INSTRUCTIONS
The goal here is to visualize some aspect of your dataset in order to help users understand it. In addition to helping users of your dataset understand the dataset, an additional goal is to impress!

Please demonstrate any data preprocessing or reshaping required for your visualization(s).

https://www.reddit.com/r/dataisbeautiful/ for inspiration.
DATA PROVIDER INSTRUCTIONS -->

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

Lets plot the intensity of the data in the UV dimensions. We can do this for all of the data by selecting a `slice(0, row_offsets[-1])` or the first chunk of it by selecting `slice(0, row_offsets[0])`.

In [ ]:
combined = xarray.concat(datasets, dim="row")
row_offsets = np.cumsum(combined.chunks["row"])
combined = combined.isel(row=slice(0, row_offsets[0]))


In [ ]:

intensity = da.absolute(combined.DATA.data[:, :, 0] * combined.WEIGHT_SPECTRUM.data[:, :, 0] * (combined.FLAG.data[:, :, 0] != 0))
render_ds = xarray.Dataset({
  "u": (("row",), combined.UVW.data[:, 0]),
  "v": (("row",), combined.UVW.data[:, 1]),
  "intensity": (("row", "chan"), intensity),
})

df = render_ds.to_dask_dataframe(dim_order=("row", "chan"))

with ProgressBar():
  canvas = datashader.Canvas()
  points = canvas.points(df, "u", "v", agg=datashader.mean("intensity"))
  plot = transfer_functions.shade(points)


In [ ]:
plot

<!-- DATA PROVIDER INSTRUCTIONS
This section is less prescriptive / freeform than previous sections. The goal here is to show an opinionated example of answering a question using your data. The scale of your dataset may preclude a full example, and so feel free to limit the scope of this example (e.g. work on a subset of data). Users should be able to replicate your example in this notebook, and get a sense of how they would scale up.

A "toy" example is better than no example.

Ideally, your example would:
1. Transmit some of your domain & dataset experience to the reader, drawing on your own work as much as possible
2. Provide a jumping off point for users to extend your work, and do novel work of their own.

DATA PROVIDER INSTRUCTIONS -->

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

EXAMPLE - REPLACE

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Praesent sem enim, finibus vel leo vel, blandit interdum tortor. In lectus dolor, congue eu odio vel, euismod facilisis lacus. Vestibulum aliquet, quam in rhoncus tempus, arcu massa suscipit lacus, nec fermentum quam justo nec lacus. Duis id leo fermentum ante tempor pulvinar eu vitae ligula. Cras feugiat vel ligula sit amet lacinia. Mauris sit amet sem vestibulum ligula volutpat iaculis in eu velit. Sed turpis magna, porta ac nisi vitae, maximus volutpat mi. Vestibulum mattis est eros, nec pellentesque nisi iaculis sed. 

EXAMPLE - REPLACE

<!-- DATA PROVIDER INSTRUCTIONS
This section is, like the previous one, intended to be freeform / non-prescriptive. The goal here is to provide a challenge to the community to do something novel with your dataset. That can either be novel in terms of the task, or novel in terms of methodological or computational approach.

Another way to consider this section, is as a wishlist. If you were less constrained by time, cost, skill, etc., what would you like to see achieved using these data? 

The challenge should, however, be somewhat realistic. A challenge that assumes e.g. original data collection, is likely to go unanswered.
DATA PROVIDER INSTRUCTIONS -->

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

EXAMPLE - REPLACE

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Praesent sem enim, finibus vel leo vel, blandit interdum tortor. In lectus dolor, congue eu odio vel, euismod facilisis lacus. Vestibulum aliquet, quam in rhoncus tempus, arcu massa suscipit lacus, nec fermentum quam justo nec lacus. Duis id leo fermentum ante tempor pulvinar eu vitae ligula. Cras feugiat vel ligula sit amet lacinia. Mauris sit amet sem vestibulum ligula volutpat iaculis in eu velit. Sed turpis magna, porta ac nisi vitae, maximus volutpat mi. Vestibulum mattis est eros, nec pellentesque nisi iaculis sed. 

EXAMPLE - REPLACE

# DATA PROVIDER: PLEASE REMEMBER TO CLEAR ALL OUTPUTS BEFORE COMMITTING TO YOUR GITHUB REPOSITORY